In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
columns = ['sex', 'Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight', 'Viscera weight', 'Shell weight', 'Rings']
file_data = pd.read_csv('data.csv', header=None, names=columns)

In [ ]:
value_counts = file_data['sex'].value_counts(normalize=True) * 100
data = {
    'category': ['Male', 'Infant', 'Female'],
    'count': file_data['sex'].value_counts(),
    '%': round(value_counts, 2)
}

df = pd.DataFrame(data)
df.set_index('category', inplace=True)
df.index.name = None
display(df)


In [ ]:
stats = {}
for column in file_data.columns[1:]:
    stats[column] = {
        'mean': file_data[column].mean(),
        'std': file_data[column].std(),
        'min': file_data[column].min(),
        '25%': file_data[column].quantile(0.25),
        '50%': file_data[column].quantile(0.5),
        '75%': file_data[column].quantile(0.75),
        'max': file_data[column].max(),
    }

df = pd.DataFrame(stats)
df.index.name = None
df = df.T
display(df)


In [ ]:
sns.barplot(x='category', y='count', data=data)
plt.title('Count by Category')
display(plt.gcf())
plt.clf()


In [ ]:
sns.set(style='whitegrid')

fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(10, 10))
axes = axes.flatten()
palette = sns.color_palette("viridis", len(file_data.columns))

for i, column in enumerate(file_data.columns[1:]):
    sns.histplot(file_data[column], bins=50, color=palette[i], ax=axes[i])
    axes[i].set_title(column)

plt.tight_layout()
display(fig)
plt.clf()


In [ ]:
fig, axes = plt.subplots(nrows=14, ncols=2, figsize=(100, 100))
axes = axes.flatten()
count = 0

for i, column_i in enumerate(file_data.columns[1:]):
    for j, column_j in enumerate(file_data.columns[2 + i:], start=i + 1):
        sns.scatterplot(x=file_data[column_i], y=file_data[column_j], hue=file_data[column_i], ax=axes[count])
        axes[count].set_title(f'{column_i} vs {column_j}')
        count += 1

plt.subplots_adjust(hspace=0.4, wspace=0.4)
display(fig)
plt.clf()


In [ ]:
df_corr = pd.DataFrame(file_data.iloc[:, 1:]).corr()
display(df_corr)


In [ ]:
plt.figure(figsize=(10, 10))
sns.heatmap(df_corr, cmap='viridis', vmin=0, vmax=1)
plt.title('Correlation Heatmap')
plt.tight_layout()
display(plt.gcf())
plt.clf()


In [ ]:
sns.regplot(x='Length', y='Diameter', data=file_data, scatter_kws={'s': 10}, ci=None)
plt.title('Regression Plot')
display(plt.gcf())
plt.clf()


In [ ]:
summary_stats = []

for col in file_data.columns[1:]:
    stats = file_data.groupby('sex')[col].agg(
        mean='mean',
        std='std',
        min='min',
        q1=lambda x: x.quantile(0.25),
        q2='median',
        q3=lambda x: x.quantile(0.75),
        max='max'
    ).reset_index()

    for _, row in stats.iterrows():
        summary_stats.append({
            'Feature': col,
            'Sex': row['sex'],
            'mean': row['mean'],
            'std': row['std'],
            'min': row['min'],
            '25%': row['q1'],
            '50%': row['q2'],
            '75%': row['q3'],
            'max': row['max']
        })

summary_df = pd.DataFrame(summary_stats)
summary_df = summary_df.set_index(['Feature', 'Sex'])
display(summary_df)


In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(20, 20))
axes = axes.flatten()

for i, name in enumerate(file_data.columns[1:]):
    plot123 = file_data.boxplot(column=name, by='sex', ax=axes[i])
    plot123.set_title(name)

plt.tight_layout()
display(fig)
plt.clf()
